# 04 — Full model fine-tuning

**Purpose:** Update all model parameters using the same examples, chat loss, optimizer family, and evaluation rules as LoRA.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

**IMPORTANT MEMORY NOTE:** Full training of 0.5B requires model weights, gradients, and optimizer state. Adafactor is selected **for both LoRA and full** to reduce optimizer-memory pressure and keep the optimizer family controlled. It is *not* guaranteed to fit T4 in every session. Start with the 2-step pilot in notebook 00 and batch size 1.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU'
TASK='sib200'; LANG='ur'; SIZES=[10]; SEEDS=[42]
MODEL_OVERRIDE=None # optionally an English-trained merged checkpoint, same as LoRA notebook
DRY_RUN=False
for seed in SEEDS:
    for n in SIZES:
        run_name=f'{TASK}_full_{LANG}_n{n}_s{seed}' + ('_anchor' if MODEL_OVERRIDE else '')
        if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
            print('Already complete:',run_name);continue
        if DRY_RUN: print('Would run',run_name); continue
        try:
            r=train_run('full',TASK,n,seed,run_name=run_name,lang=LANG,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),'GPU GiB:',r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            print('FULL TRAIN OOM. Stop this kernel; reduce max_length to 128 or use more memory.')
            print('Do not claim a full-tuning result until a completed checkpoint is saved.')
            raise

When you have successful results at 10/class, add 25 and 50/class. Compare the same sizes and seeds as LoRA. Learning rates can differ between methods, but tune them fairly using the validation set, then freeze settings before final test evaluation.